### Packages Imported

In [0]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.impute import SimpleImputer
import seaborn as sns 
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 

### converting pyspark dataframe into pandas dataframe using toPandas()

In [0]:
df_cleaned = spark.read.table("workspace.default.batsmen_statistics_table_cleaned")
df_pandas = df_cleaned.toPandas()


In [0]:
df_pandas.head(5)

### Variance Threshold is a feature selection method for numerical features using this we can eliminate columns which has little variation of data over the span of the dataset/ data frame / database table  

In [0]:
vt = VarianceThreshold(threshold=1)

In [0]:
df_numerical_for_vt = df_pandas.select_dtypes(include="number")

In [0]:
vt_result = vt.fit_transform(df_numerical_for_vt)

In [0]:
len(vt.get_feature_names_out(df_numerical_for_vt.columns))

### Pearson Correlation

In [0]:
df_corr = df_numerical_for_vt.corr(method="pearson", numeric_only=True)

In [0]:
plt.figure(figsize=(18,15))
sns.heatmap(
            df_corr, 
            annot=False, 
            cmap="coolwarm", 
            center=0, 
            linewidths=0.5, 
            )
plt.title("Pearson_Correlation_plot")
plt.show()

In [0]:
vt_result = vt.fit_transform(df_numerical_for_vt)

In [0]:
len(vt.get_feature_names_out(df_numerical_for_vt.columns))

### spearman Correlation 

In [0]:
df_corr_spearman = df_numerical_for_vt.corr(method='spearman', numeric_only=True)


In [0]:
plt.figure(figsize=(18,15))
sns.heatmap(
    df_corr_spearman, 
    annot=False,
    cmap="coolwarm", 
    center=0,
    linewidths=0.5, 
    fmt=".2f"
    )
plt.title("Spearman_Correlation_plot")
plt.show()

### Removing redundant columns using correlation

In [0]:
upper = df_corr.where(
                        np.triu(np.ones(df_corr.shape), k=1).astype(bool)
                        )

df_corr_features_stacked = upper.stack().sort_values(key=abs, ascending=False)
df_corr_features_stacked_filtered = (
    df_corr_features_stacked[(abs(df_corr_features_stacked) >= 0) & (abs(df_corr_features_stacked) <= 0.40)]
)
corr_features_stacked_filtered_to_df = df_corr_features_stacked_filtered.reset_index()
corr_features_stacked_filtered_to_df.columns = ['feature_1', 'feature_2', 'corr_value']
corr_features_stacked_filtered_to_df

In [0]:
corr_features_stacked_filtered_to_df.iloc[30:40]

### Features selected using pearson correlation score and domain knowledge

#### Features selected
#### 1) Total_boundaries
#### 2) 6s_per_Ball_in_power_play
#### 3) 6s_per_Ball_in_middle_overs
#### 4) 6s_per_Ball_in_death_overs
#### 5) Balls per boundary in powerplay
#### 6) Balls per boundary in middle_overs
#### 7) Balls per boundary in death_overs
#### 8) Dot_ball_percentage_in_powerplay
#### 9) Dot_ball_percentage_in_middle_overs
#### 10) Dot_ball_percentage_in_Death_overs
#### 11) Batsman_Strike_rate_in_powerplay
#### 12) Batsman_Strike_rate_in_middle_overs
#### 13) Batsman_Strike_rate_in_Death_overs
#### 14) Batsman_average_in_powerplay
#### 15) Batsman_average_in_middle_overs
#### 16) Batsman_average_in_death_overs
#### 17) 4s_per_Ball_in_power_play
#### 18) 4s_per_Ball_in_middle_overs
#### 19) 4s_per_Ball_in_death_overs


### Imputing Null values with a constant 

In [0]:
imputer = SimpleImputer(
    strategy='constant',
    fill_value=0,
    add_indicator=True
    )

In [0]:
df_with_null_values = df_numerical_for_vt[['Dot_ball_percentage_in_powerplay',
       'Dot_ball_percentage_in_middle_overs',
       'Dot_ball_percentage_in_death_overs', 'batsman_average_in_powerplay',
       'batsman_average_in_middle_overs', 'batsman_average_in_death_overs',
       'Batsman_Strike_rate_in_powerplay',
       'Batsman_Strike_rate_in_middle_overs',
       'Batsman_Strike_rate_in_death_overs', 'Balls_per_Boundary_in_powerplay',
       'Balls_per_Boundary_in_middle_overs',
       'Balls_per_Boundary_in_death_overs', '4s_per_Ball_in_powerplay',
       '4s_per_Ball_in_middle_overs', '4s_per_Ball_in_death_overs',
       '6s_per_Ball_in_powerplay', '6s_per_Ball_in_middle_overs',
       '6s_per_Ball_in_death_overs']]

In [0]:
df_with_null_values_imputer_fit = imputer.fit(df_with_null_values)
df_with_null_values_imputer_transform = df_with_null_values_imputer_fit.transform(df_with_null_values)

In [0]:
df_with_selected_features_imputed = pd.DataFrame(
    df_with_null_values_imputer_transform,
    columns=imputer.get_feature_names_out()
)

### Scaling values col 1 

In [0]:
df_with_selected_features_imputed.columns

In [0]:
df_with_selected_features = df_with_selected_features_imputed

In [0]:
df_with_selected_features

In [0]:
df_col_Dot_ball_percentage_in_powerplay = df_with_selected_features[["Dot_ball_percentage_in_powerplay"]]

In [0]:
df_col_Dot_ball_percentage_in_powerplay.shape

In [0]:
plt.boxplot(df_col_Dot_ball_percentage_in_powerplay['Dot_ball_percentage_in_powerplay'].dropna())
plt.title('Dot_ball_percentage_in_powerplay')
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_Dot_ball_percentage_in_powerplay['Dot_ball_percentage_in_powerplay'].dropna(), bins=20)
plt.title('Dot_ball_percentage_in_powerplay')
plt.show()

In [0]:
scaler = StandardScaler()

In [0]:
scaler_fit_db_pp = scaler.fit(df_col_Dot_ball_percentage_in_powerplay)

In [0]:
scaled_df = pd.DataFrame()

In [0]:
scaled_df['Scaled_Dot_ball_percentage_in_powerplay'] = (
    scaler_fit_db_pp.transform(df_col_Dot_ball_percentage_in_powerplay)
).ravel()

In [0]:
scaled_df

### Scaling dot ball percentage in middle orders col 2 

In [0]:
df_col_Dot_ball_percentage_in_middle_overs = df_with_selected_features[["Dot_ball_percentage_in_middle_overs"]]

In [0]:
df_col_Dot_ball_percentage_in_middle_overs

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_Dot_ball_percentage_in_middle_overs.dropna())
plt.title("Dot_ball_percentage_in_middle_overs")
plt.show()


In [0]:
plt.figure(figsize=(15, 12))
plt.hist(df_col_Dot_ball_percentage_in_middle_overs.dropna(), bins=20)
plt.title("Distribution of Dot_ball_percentage_in_middle_overs")
plt.show()

In [0]:

df_dbp_in_mo_not_na_scaler = PowerTransformer(method ='yeo-johnson', standardize=True)

In [0]:
scaler_dpo_in_mo_not_na_fit = df_dbp_in_mo_not_na_scaler.fit(df_col_Dot_ball_percentage_in_middle_overs)

In [0]:
scaler_dpo_in_mo_not_na_transform = scaler_dpo_in_mo_not_na_fit.transform(df_col_Dot_ball_percentage_in_middle_overs)

In [0]:
scaler_dpo_in_mo_not_na_transform

In [0]:
scaled_df['Scaled_Dot_ball_percentage_in_middle_overs'] = scaler_dpo_in_mo_not_na_transform.ravel()

In [0]:
scaled_df

In [0]:
scaled_df[["Scaled_Dot_ball_percentage_in_powerplay"]].min()

### Scaling dot ball percentage in death overs col 3 

In [0]:
df_col_Dot_ball_percentage_in_death_overs = df_with_selected_features[["Dot_ball_percentage_in_death_overs"]]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_Dot_ball_percentage_in_death_overs.dropna())
plt.title("Boxplot of Dot_ball_percentage_in_death_overs")
plt.show()

In [0]:
plt.figure(figsize=(12,10))
plt.hist(df_col_Dot_ball_percentage_in_death_overs.dropna())
plt.title("Histogram of Dot_ball_percentage_in_death_overs")
plt.show()

In [0]:
power_transformer_scaler_col_3 = PowerTransformer(method='yeo-johnson', standardize=True) 

In [0]:
scaler_dpo_in_do_fit = (
power_transformer_scaler_col_3.fit(df_col_Dot_ball_percentage_in_death_overs)
)

scaler_dpo_in_do_transform = scaler_dpo_in_do_fit.transform(df_col_Dot_ball_percentage_in_death_overs)


In [0]:
scaled_df["Scaled_Dot_ball_percentage_in_death_overs"] = scaler_dpo_in_do_transform.ravel()

In [0]:
scaled_df

### Scaling batsmen average in powerplay col 4 

In [0]:
df_col_batsman_average_in_powerplay = df_with_selected_features[['batsman_average_in_powerplay']]

In [0]:
plt.figure(figsize=(15, 12))
plt.boxplot(df_col_batsman_average_in_powerplay.dropna())
plt.title("Average of batsman_average_in_powerplay")
plt.show()


In [0]:
plt.figure(figsize=(15, 12))
plt.hist(df_col_batsman_average_in_powerplay.dropna())
plt.title("Histogram of batsman_average_in_powerplay")
plt.show()

In [0]:
power_transformer_scaler_col_4 = PowerTransformer(method='yeo-johnson', standardize=True)

In [0]:
scaler_ba_in_pp_fit = power_transformer_scaler_col_4.fit(df_col_batsman_average_in_powerplay)

In [0]:
scaler_ba_in_pp_transform = scaler_ba_in_pp_fit.transform(df_col_batsman_average_in_powerplay)
scaled_df["Scaled_batsman_average_in_powerplay"]= scaler_ba_in_pp_transform.ravel()

In [0]:
scaled_df

In [0]:
plt.figure(figsize=(12, 15))
plt.hist(scaled_df["Scaled_batsman_average_in_powerplay"])
plt.title("distribution of scaled batsman_average_in_powerplay")
plt.show()


###Scaling values batsmen average in middle overs  col 5

In [0]:
df_with_selected_features[['batsman_average_in_middle_overs']].iloc[0:1]

In [0]:
df_col_batsman_avg_in_middle_overs = df_with_selected_features[['batsman_average_in_middle_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_batsman_avg_in_middle_overs.dropna())
plt.title('Batsman Average in Middle Overs')
plt.xticks([1], ['batsman_average_in_middle_overs'])
plt.show()


In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_batsman_avg_in_middle_overs.dropna())
plt.title('Batsman Average in Middle Overs')
plt.show()

In [0]:
robust_scaler_col_5 = RobustScaler() 

In [0]:
scaler_ba_in_mo_fit = robust_scaler_col_5.fit(df_col_batsman_avg_in_middle_overs)
scaler_ba_in_mo_transform = scaler_ba_in_mo_fit.transform(df_col_batsman_avg_in_middle_overs)

In [0]:
scaled_df['Scaled_batsman_average_in_middle_overs'] = scaler_ba_in_mo_transform.ravel()


###Scaling values batsmen average in death overs col 6

In [0]:
df_col_batsman_avg_in_death_overs = df_with_selected_features[['batsman_average_in_death_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_batsman_avg_in_death_overs.dropna())
plt.title('Batsman Average in Death')
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_batsman_avg_in_death_overs.dropna())
plt.title('Batsman Average in Death')
plt.show()

In [0]:
robust_scaler_col_6 =RobustScaler()
scaler_ba_in_do_fit = robust_scaler_col_6.fit(df_col_batsman_avg_in_death_overs)
scaler_ba_in_do_transform = scaler_ba_in_do_fit.transform(df_col_batsman_avg_in_death_overs)
scaled_df['Scaled_batsman_average_in_death_overs'] = scaler_ba_in_do_transform.ravel()

In [0]:
scaled_df

In [0]:
df_with_selected_features

In [0]:
df_col_Batsman_Strike_rate_in_powerplay = df_with_selected_features[['Batsman_Strike_rate_in_powerplay']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_Batsman_Strike_rate_in_powerplay.dropna())
plt.title("Batsman Strike Rate in Powerplay")
plt.show()

In [0]:
plt.figure(figsize=(15, 12))
plt.hist(df_col_Batsman_Strike_rate_in_powerplay.dropna())
plt.title("Batsman Strike Rate in Powerplay")
plt.show()

In [0]:
robust_scaler_col_7 = RobustScaler()

In [0]:
scaler_bsr_in_pp_fit = robust_scaler_col_7.fit(df_col_Batsman_Strike_rate_in_powerplay)
scaler_bsr_in_pp_transform = scaler_bsr_in_pp_fit.transform(df_col_Batsman_Strike_rate_in_powerplay)
scaled_df['Scaled_Batsman_Strike_rate_in_powerplay'] = scaler_bsr_in_pp_transform.ravel()

###Scaling values batsmen strike rate in middle overs col 8

In [0]:
df_col_Batsman_Strike_rate_in_middle_overs = df_with_selected_features[['Batsman_Strike_rate_in_middle_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_Batsman_Strike_rate_in_middle_overs.dropna())
plt.title("Batsman Strike Rate in Middle Overs")
plt.show()

In [0]:
plt.figure(figsize=(15, 12))
plt.hist(df_col_Batsman_Strike_rate_in_middle_overs.dropna())
plt.title("Batsman Strike Rate in Middle Overs")
plt.show()

In [0]:
standard_scaler_col_8 = StandardScaler()
scaler_bsr_in_mo_fit = standard_scaler_col_8.fit(df_col_Batsman_Strike_rate_in_middle_overs)
scaler_bsr_in_mo_transform = scaler_bsr_in_mo_fit.transform(df_col_Batsman_Strike_rate_in_middle_overs)
scaled_df['Scaled_Batsman_Strike_rate_in_middle_overs'] = scaler_bsr_in_mo_transform.ravel()

###Scaling values batsmen strike rate in death overs col 9

In [0]:
df_col_batsman_strike_rate_in_death_overs = df_with_selected_features[['Batsman_Strike_rate_in_death_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_batsman_strike_rate_in_death_overs.dropna())
plt.title("batsman_sr_outlier")
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_batsman_strike_rate_in_death_overs.dropna())
plt.title("batsman_sr_outlier")
plt.show()

In [0]:
power_transformer_scaler_col_9 = PowerTransformer(method='yeo-johnson', standardize=True)
scaler_bsr_in_death_overs_fit = power_transformer_scaler_col_9.fit(df_col_batsman_strike_rate_in_death_overs)
scaler_bsr_in_do_transform = scaler_bsr_in_death_overs_fit.transform(df_col_batsman_strike_rate_in_death_overs)
scaled_df['Scaled_Batsman_Strike_rate_in_death_overs'] = scaler_bsr_in_do_transform.ravel()

In [0]:
scaled_df

###Scaling values batsmen strike rate in death overs col 10

In [0]:
df_col_balls_per_boundary_in_powerplay = df_with_selected_features[['Balls_per_Boundary_in_powerplay']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_balls_per_boundary_in_powerplay.dropna())
plt.title("Balls_per_Boundary_in_powerplay")
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_balls_per_boundary_in_powerplay.dropna())
plt.title("balls_per_boundary_distribution")
plt.show()

In [0]:
robust_scaler_col_10 = RobustScaler()
scaler_bpb_in_powerplay_fit = robust_scaler_col_10.fit(df_col_balls_per_boundary_in_powerplay)
scaler_bpb_in_powerplay_transform = scaler_bpb_in_powerplay_fit.transform(df_col_balls_per_boundary_in_powerplay)

In [0]:
scaled_df["scaled_bpb_in_powerplay"] = scaler_bpb_in_powerplay_transform.ravel()

###Scaling values ball per boundary in middle overs col 11

In [0]:
df_col_balls_per_boundary_in_middle_overs = df_with_selected_features[['Balls_per_Boundary_in_middle_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_balls_per_boundary_in_middle_overs.dropna())
plt.title("Balls_per_Boundary_in_middle_overs")
plt.show()

In [0]:
plt.figure(figsize=(15, 12))
plt.hist(df_col_balls_per_boundary_in_middle_overs.dropna())
plt.title("balls_per_boundary_in_middle_overs_distribution")
plt.show()

In [0]:
RobustScaler_col_11 = RobustScaler()
scaler_bpb_in_middle_overs_fit = RobustScaler_col_11.fit(df_col_balls_per_boundary_in_middle_overs)
scaler_bpb_in_middle_overs_transform = scaler_bpb_in_middle_overs_fit.transform(df_col_balls_per_boundary_in_middle_overs)
scaled_df["scaled_bpb_in_middle_overs"] = scaler_bpb_in_middle_overs_transform.ravel()

###Scaling values batsmen balls per boundary in death overs col 12

In [0]:
df_col_balls_per_boundary_in_death_overs = df_with_selected_features[['Balls_per_Boundary_in_death_overs']]

In [0]:
plt.figure(figsize=(15, 12))
plt.boxplot(df_col_balls_per_boundary_in_death_overs.dropna())
plt.title("Balls_per_Boundary_in_death_overs")
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_balls_per_boundary_in_death_overs.dropna())
plt.title("balls_per_boundary_in_death_overs_distribution")
plt.show()

In [0]:
RobustScaler_col_12 = RobustScaler()
scaler_bpb_in_death_overs_fit = RobustScaler_col_12.fit(df_col_balls_per_boundary_in_death_overs)
scaler_bpb_in_death_overs_transform = scaler_bpb_in_death_overs_fit.transform(df_col_balls_per_boundary_in_death_overs)

In [0]:
scaled_df["scaled_bpb_in_death_overs"] = scaler_bpb_in_death_overs_transform.ravel()


###Scaling values batsmen 4s per ball in powerplay col 13

In [0]:
df_col_fours_per_ball_in_powerplay = df_with_selected_features[['4s_per_Ball_in_powerplay']]

In [0]:
plt.figure(figsize=(15, 12))
plt.boxplot(df_col_fours_per_ball_in_powerplay.dropna())
plt.title("4s_per_Ball_in_powerplay")
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_fours_per_ball_in_powerplay.dropna())
plt.title("4s_per_Ball_in_powerplay_distribution")
plt.show()

In [0]:
RobustScaler_col_13 = RobustScaler()
scaler_4s_per_ball_in_powerplay_fit = RobustScaler_col_13.fit(df_col_fours_per_ball_in_powerplay)
scaler_4s_per_ball_in_powerplay_transform = scaler_4s_per_ball_in_powerplay_fit.transform(df_col_fours_per_ball_in_powerplay)

In [0]:
scaled_df["scaled_4s_per_ball_in_powerplay"] = scaler_4s_per_ball_in_powerplay_transform.ravel()

###Scaling values batsmen 4s per ball in middle overs col 14

In [0]:
df_col_fours_per_ball_in_middle_overs = df_with_selected_features[['4s_per_Ball_in_middle_overs']]

In [0]:
plt.figure(figsize=(15,12))
plt.boxplot(df_col_fours_per_ball_in_middle_overs.dropna())
plt.title("4s_per_Ball_in_middle_overs_Outlier_plot")
plt.show()

In [0]:
plt.figure(figsize=(15,12))
plt.hist(df_col_fours_per_ball_in_middle_overs.dropna())
plt.title("4s_per_Ball_in_middle_overs_distribution")
plt.show()

In [0]:
RobustScaler_col_14 = RobustScaler()
scaler_4s_per_ball_in_middle_overs_fit = RobustScaler_col_14.fit(df_col_fours_per_ball_in_middle_overs)
scaler_4s_per_ball_in_middle_overs_transform = scaler_4s_per_ball_in_middle_overs_fit.transform(df_col_fours_per_ball_in_middle_overs)


In [0]:
scaled_df["scaled_4s_per_ball_in_middle_overs"] = scaler_4s_per_ball_in_middle_overs_transform.ravel()

###Scaling values batsmen 4s per ball in death overs col 15

In [0]:
df_col_fours_per_ball_in_death_overs = df_with_selected_features[['4s_per_Ball_in_death_overs']]

In [0]:
plt.figure(figsize=((15, 12)))
sns.boxplot(df_col_fours_per_ball_in_death_overs.dropna())
plt.title('Boxplot of 4s per Ball in Death Overs')
plt.xlabel('4s per Ball in Death Overs')
plt.show()

In [0]:
plt.figure(figsize=((15, 12)))
sns.histplot(df_col_fours_per_ball_in_death_overs.dropna())
plt.title('Boxplot of 4s per Ball in Death Overs')
plt.xlabel('distribution of 6s per Ball in Death Overs')
plt.show()

In [0]:
RobustScaler_col_15 = RobustScaler()
scaler_4s_per_ball_in_death_overs_fit = RobustScaler_col_15.fit(df_col_fours_per_ball_in_death_overs)
scaler_4s_per_ball_in_death_overs_transform = scaler_4s_per_ball_in_death_overs_fit.transform(df_col_fours_per_ball_in_death_overs)

In [0]:
scaled_df["scaled_4s_per_ball_in_death_overs"] = scaler_4s_per_ball_in_death_overs_transform.ravel()

###Scaling values batsmen 6s per ball in powerplay col 16

In [0]:
df_col_sixers_per_ball_in_powerplay = df_with_selected_features[['6s_per_Ball_in_powerplay']]

In [0]:
plt.figure(figsize=((15, 12)))
sns.boxplot(df_col_sixers_per_ball_in_powerplay.dropna())
plt.title('Sixers per ball in powerplay')
plt.xlabel('Sixers per ball in powerplay')
plt.show()

In [0]:
plt.figure(figsize=((15, 12)))
sns.histplot(df_col_sixers_per_ball_in_powerplay)
plt.title('Sixers per ball in powerplay')
plt.xlabel('distribution of Sixers per ball in powerplay')
plt.show()

In [0]:
RobustScaler_col_16 = RobustScaler()
scaler_6s_per_ball_in_powerplay_fit = RobustScaler_col_16.fit(df_col_sixers_per_ball_in_powerplay)
scaler_6s_per_ball_in_powerplay_transform = scaler_6s_per_ball_in_powerplay_fit.transform(df_col_sixers_per_ball_in_powerplay)


In [0]:
scaled_df["scaled_6s_per_ball_in_powerplay"] = scaler_6s_per_ball_in_powerplay_transform.ravel()

In [0]:
scaled_df

###Scaling values batsmen 6s per ball in middle overs col 17

In [0]:
df_col_sixers_per_ball_in_middle_overs = df_with_selected_features[['6s_per_Ball_in_middle_overs']]

In [0]:
plt.figure(figsize=((15, 25)))
sns.boxplot(df_col_sixers_per_ball_in_middle_overs.dropna())
plt.title('Outliers Sixers per ball in middle overs')
plt.xlabel('Sixers per ball in middle overs')
plt.show() 

In [0]:
plt.figure(figsize=((15, 12)))
sns.histplot(df_col_sixers_per_ball_in_middle_overs)
plt.title('Distribution of Sixers per ball in middle overs')
plt.xlabel('Sixers per ball in middle overs')
plt.show()

In [0]:
RobustScaler_col_17 = RobustScaler()
scaler_6s_per_ball_in_middle_overs_fit = RobustScaler_col_17.fit(df_col_sixers_per_ball_in_middle_overs)
scaler_6s_per_ball_in_middle_overs_transform = scaler_6s_per_ball_in_middle_overs_fit.transform(df_col_sixers_per_ball_in_middle_overs)

In [0]:
scaled_df["scaled_6s_per_ball_in_middle_overs"] = scaler_6s_per_ball_in_middle_overs_transform.ravel()

In [0]:
scaled_df

###Scaling values batsmen 6s per ball in death overs col 18

In [0]:
df_col_sixers_per_ball_in_death_overs = df_with_selected_features[['6s_per_Ball_in_death_overs']]

In [0]:
plt.figure(figsize=((15, 25)))
sns.boxplot(df_col_sixers_per_ball_in_death_overs.dropna())
plt.title('Outliers Sixers per ball in death overs')
plt.xlabel('Sixers per ball in death overs')
plt.show() 

In [0]:
plt.figure(figsize=((15, 12)))
sns.histplot(df_col_sixers_per_ball_in_death_overs)
plt.title('Distribution of Sixers per ball in death overs')
plt.xlabel('Sixers per ball in death overs')
plt.show()

In [0]:
RobustScaler_col_18 = RobustScaler()
scaler_6s_per_ball_in_death_overs_fit = RobustScaler_col_18.fit(df_col_sixers_per_ball_in_death_overs)
scaler_6s_per_ball_in_death_overs_transform = scaler_6s_per_ball_in_death_overs_fit.transform(df_col_sixers_per_ball_in_death_overs)


In [0]:
scaled_df["scaled_6s_per_ball_in_death_overs"] = scaler_6s_per_ball_in_death_overs_transform.ravel()


In [0]:
scaled_df

### Applying cosine similarity 

In [0]:
df_with_selected_features

In [0]:
scaled_df

In [0]:
indicator_columns = ['missingindicator_Dot_ball_percentage_in_powerplay',
       'missingindicator_Dot_ball_percentage_in_middle_overs',
       'missingindicator_Dot_ball_percentage_in_death_overs',
       'missingindicator_batsman_average_in_powerplay',
       'missingindicator_batsman_average_in_middle_overs',
       'missingindicator_batsman_average_in_death_overs',
       'missingindicator_Batsman_Strike_rate_in_powerplay',
       'missingindicator_Batsman_Strike_rate_in_middle_overs',
       'missingindicator_Batsman_Strike_rate_in_death_overs',
       'missingindicator_Balls_per_Boundary_in_powerplay',
       'missingindicator_Balls_per_Boundary_in_middle_overs',
       'missingindicator_Balls_per_Boundary_in_death_overs',
       'missingindicator_4s_per_Ball_in_powerplay',
       'missingindicator_4s_per_Ball_in_middle_overs',
       'missingindicator_4s_per_Ball_in_death_overs',
       'missingindicator_6s_per_Ball_in_powerplay',
       'missingindicator_6s_per_Ball_in_middle_overs',
       'missingindicator_6s_per_Ball_in_death_overs']

In [0]:
scaled_df[indicator_columns] = df_with_selected_features[indicator_columns]

In [0]:
scaled_df.isnull().sum()

In [0]:
cosine_matrix = cosine_similarity(scaled_df)

In [0]:
cosine_similarity_df = pd.DataFrame(
    cosine_matrix, 
    index=df_pandas['batter'],
    columns=df_pandas['batter']
)

In [0]:
cosine_similarity_df

In [0]:
target_player = 'T Stubbs'

similar_players = (
    cosine_similarity_df[target_player].drop(target_player)
    .sort_values(ascending=False)
)

print(similar_players.head(10))